# The "News" Calculation: Plotly Waterfall Chart (Live Execution)

This notebook demonstrates how to execute the DFM `update_nowcast` against the real benchmark datasets housed in `data/US/` and cleanly pipe the resulting Kalman impacts straight into a Plotly visualization.

In [ ]:
import os
import numpy as np
import pandas as pd
from dfm_sp import load_data
from dfm_sp import update_nowcast
from dfm_sp import Options
from dfm_sp import run_with_options, run
from dfm_sp import plot_news_waterfall


### 1. Execute Baseline DFM
We load the historical matrices using the standard NY Fed example target variables and compute the original baseline via the EM-Algorithm/Kalman Smoothing bounds.

In [2]:
# Define targets
series = 'GDPC1'
period = '2016q4'
vintage_old  = '2016-12-16'
vintage_new  = '2016-12-23'

options_baseline = Options(
    vintage=vintage_old,
    max_iter=5000, 
    use_cache=True
)

print(f"Executing Baseline for {vintage_old}...")
Spec, X_old, Time_old, Z_old = run_with_options(options_baseline)
ResObject = run(X_old, Spec, options_baseline)
Res = ResObject.result

Executing Baseline for 2016-12-16...

 Table 1: Model specification 

              SeriesID                   SeriesName                 Units  \
0               PAYEMS           Payroll Employment  Thousands of Persons   
1               JTSJOL                 Job Openings             Thousands   
2             CPIAUCSL         Consumer Price Index                 Index   
3              DGORDER         Durable Goods Orders           $, Millions   
4                RSAFS                 Retail Sales           $, Millions   
5               UNRATE            Unemployment Rate                     %   
6                HOUST               Housing Starts    Thousands of Units   
7               INDPRO        Industrial Production                 Index   
8              DSPIC96              Personal Income   Chained $, Billions   
9              BOPTEXP                      Exports           $, Millions   
10             BOPTIMP                      Imports           $, Millions   
11    

### 2. Load the "New" Data Vintage and Calculate News
By passing `X_old` vs `X_new`, `update_nowcast` detects exactly which data cells transitioned from `np.nan` to concrete releases. It then weights those surprises mathematically against what the Kalman predicted.

In [3]:
datafile_new = os.path.join('data', options_baseline.country, vintage_new + '.xls')
X_new, Time, _ = load_data(datafile_new, Spec, options_baseline.sample_start)

y_old, y_new, news_table, data_released = update_nowcast(
    X_old, X_new, Time, Spec, Res, series, period, vintage_old, vintage_new
)

# Isolate only the actual news triggers
real_impacts = news_table.iloc[np.where(data_released)[0], :]
display(real_impacts)

2000-01-01 00:00:00

 Nowcast Update: 2016-12-23

 Nowcast for: Real Gross Domestic Product (Percent Change (Annual Rate)), 2016Q4

 Nowcast Impact Decomposition
 Note: The displayed output is subject to rounding error


              2016-12-16 nowcast:              2.47088
      Impact from data revisions:      0.12903
       Impact from data releases:      -0.10551
                                     +_________
                    Total impact:      0.02352
              2016-12-23 nowcast:              2.49439

  Nowcast Detail Table 

          Forecast    Actual    Weight    Impact
DGORDER  -1.901732 -4.598821  0.007450 -0.020093
DSPIC96   0.177420 -0.050908  0.011193 -0.002556
PCEPILFE  0.140903  0.004467  0.239987 -0.032743
PCEPI     0.166527  0.041284  0.274312 -0.034356
PCEC96    0.188004  0.144587  0.363102 -0.015765


,Forecast,Actual,Weight,Impact
DGORDER,-1.901732,-4.598821,0.007450,-0.020093
DSPIC96,0.177420,-0.050908,0.011193,-0.002556
PCEPILFE,0.140903,0.004467,0.239987,-0.032743
PCEPI,0.166527,0.041284,0.274312,-0.034356
PCEC96,0.188004,0.144587,0.363102,-0.015765


### 3. Generate Plotly Waterfall Visualization

In [4]:
fig = plot_news_waterfall(
    news_table=real_impacts, 
    y_old=y_old, 
    y_new=y_new,
    vintage_old=vintage_old,
    vintage_new=vintage_new,
    target_series="Real GDP Growth (Q4)"
)

fig.show()